# Lab 3.2.2 — Prepare data for a machine learning model

**Hands-on objective:** perform the applicable data-preparation steps from section 3.2.1 and produce data for a supervised classification model.

This notebook follows **predict → act → observe → explain**. It is the only chronological workspace. Write in each empty answer cell before running the code that follows it. Open `learning_log.md` only when the final checkpoint directs you there.

## Learning agreement and perimeter

- Code provides evidence; your explanation provides the learning.
- Open hints progressively and only after making an attempt.
- After splitting, inspect feature values from training data only.
- The final supplied section trains one fixed model for later exercises, but this lab does **not** calculate metrics, tune the model, compare models, or claim production readiness.
- The dataset contains historical demographic and financial-status fields. Using them here supports a bounded educational benchmark; it does not endorse real-world profiling.

You will finish with four handoff artifacts, not with a performance score.

In [ ]:
# Preflight: this cell should run without edits.
import hashlib
import json
import platform
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

LAB_ROOT = Path.cwd()
RAW_DATA_PATH = LAB_ROOT / "data" / "raw" / "bank-additional-full.csv"
SOURCE_SHA256 = "74ADFC578BF77A7FF4BB1BA4A9F8709D9E3C6907342959C2C8416847E0AFB4D8"

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

## 1. Acquire with provenance, not just a filename

The raw file is UCI's `bank-additional-full.csv`, preserved unchanged. It represents historical direct-marketing calls by a Portuguese bank. The target `y` records whether the client subscribed to a term deposit.

Our intended prediction moment is **immediately before a scheduled call begins**. Every proposed model input must be available at that moment.

**STOP — answer below before verifying the file:**

1. What is one example, the label, and the two classes?
2. Why is this supervised classification?
3. What does verifying a source hash establish? What can it not establish about data quality?
4. Which syllabus data-preparation activity does acquiring and verifying this supplied file support?

> **Your answer:**  
> 
> 


In [ ]:
# Source-integrity and schema checks: run without edits.
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as source_file:
        for block in iter(lambda: source_file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()

assert RAW_DATA_PATH.exists(), f"Missing raw dataset: {RAW_DATA_PATH}"
actual_hash = file_sha256(RAW_DATA_PATH)
assert actual_hash == SOURCE_SHA256, (
    "The raw file does not match the documented UCI artifact. "
    f"Expected {SOURCE_SHA256}, got {actual_hash}."
)

raw = pd.read_csv(RAW_DATA_PATH, sep=";")
assert raw.shape == (41_188, 21), f"Unexpected raw shape: {raw.shape}"
assert raw["y"].value_counts().to_dict() == {"no": 36_548, "yes": 4_640}
print("Verified canonical UCI file.")
print("Shape:", raw.shape)
print("Target counts:", raw["y"].value_counts().to_dict())

## 2. Inventory before changing anything

A preparation decision should follow evidence and source semantics. First inspect column names, sample records, pandas data types, literal `unknown` values, ranges, and exact row matches.

**STOP:** predict why pandas will report no null values even though UCI's included documentation says some information is missing.

> **Your prediction:**  
> 


In [ ]:
print(raw.head(3).to_string(index=False))
print("\nData types:")
print(raw.dtypes.to_string())

unknown_counts = (raw == "unknown").sum()
unknown_counts = unknown_counts[unknown_counts > 0].sort_values(ascending=False)
exact_matches_beyond_first = int(raw.duplicated(keep="first").sum())

print("\nPandas null cells:", int(raw.isna().sum().sum()))
print("Literal 'unknown' counts:")
print(unknown_counts.to_string())
print("\nExact matching rows beyond the first:", exact_matches_beyond_first)
print("pdays most frequent values:", raw["pdays"].value_counts().head().to_dict())

### Interpret the inventory

**STOP — answer precisely:**

1. Why is literal `unknown` semantically different from pandas `NaN` here?
2. Does exact equality across all published columns prove two rows are the same real-world call? What missing field prevents that conclusion?
3. Is `pdays=999` evidence of an extreme wait, or a coded state? Use `data/raw/bank-additional-names.txt`, not intuition.
4. Name one numerical and one categorical input. Why should a number-looking value not automatically be treated as numerical?

<details><summary>Source nudge</summary>The source contains no client or call identifier, and its documentation assigns a special meaning to 999.</details>

> **Your answer:**  
> 
> 


## 3. Make structural decisions from the prediction boundary

UCI warns that one field strongly affects the target but is known only after the call. High predictive power does not make a feature valid.

**STOP:** identify that field and explain the temporal contradiction in one sentence. Then complete the four decisions below.

<details><summary>Hint 1 — availability</summary>Read the description of <code>duration</code> in the included names file.</details>
<details><summary>Hint 2 — expected decision tokens</summary>Use <code>before_scheduled_call</code>, the raw column name, the two target strings mapped to 0/1, and a Boolean for retaining exact matches.</details>

> **Your reasoning:**  
> 


In [ ]:
PREDICTION_TIME = None        # TODO: a short token describing the agreed moment
LEAKAGE_COLUMN = None         # TODO: one raw column name
TARGET_MAPPING = None         # TODO: map the two raw target strings to 0 and 1
KEEP_EXACT_MATCHES = None     # TODO: True or False, based on the available identity evidence

assert PREDICTION_TIME == "before_scheduled_call", "Use the prediction boundary defined above."
assert LEAKAGE_COLUMN == "duration", "Recheck which field exists only after the call."
assert TARGET_MAPPING == {"no": 0, "yes": 1}, "Encode the positive class as 1."
assert KEEP_EXACT_MATCHES is True, (
    "Without a source identity field, equality does not prove duplicate real-world records."
)

In [ ]:
# Deterministic structural preparation before splitting.
structured = raw.copy()
structured.insert(0, "source_row_id", np.arange(1, len(structured) + 1))
structured = structured.rename(columns=lambda name: name.replace(".", "_"))
structured = structured.rename(columns={"y": "subscribed"})
structured["subscribed"] = structured["subscribed"].map(TARGET_MAPPING)
structured = structured.drop(columns=[LEAKAGE_COLUMN])

assert structured["source_row_id"].is_unique
assert set(structured["subscribed"].unique()) == {0, 1}
assert "duration" not in structured.columns
assert len(structured) == len(raw), "The evidence did not justify deleting rows."
print("Structural preparation complete:", structured.shape)

## 4. Freeze three evidence roles before learned preparation

The supplied handoff policy is a stratified **60% training / 20% validation / 20% test** split with seed 42. Splitting now prevents EDA, imputation statistics, scaling parameters, and category learning from using validation or test feature values.

This split is scaffolding that applies section 3.2.3; you are not asked to optimize it.

**STOP:** explain why deterministic column renaming may safely happen before the split, while calculating a median for imputation should happen after it.

> **Your answer:**  
> 


In [ ]:
# Fixed suite handoff: run without edits.
SPLIT_SEED = 42
TEST_FRACTION = 0.20
VALIDATION_FRACTION_OF_REMAINDER = 0.25

development, test = train_test_split(
    structured,
    test_size=TEST_FRACTION,
    random_state=SPLIT_SEED,
    stratify=structured["subscribed"],
)
train, validation = train_test_split(
    development,
    test_size=VALIDATION_FRACTION_OF_REMAINDER,
    random_state=SPLIT_SEED,
    stratify=development["subscribed"],
)

expected_sizes = {"train": 24_712, "validation": 8_238, "test": 8_238}
actual_sizes = {"train": len(train), "validation": len(validation), "test": len(test)}
assert actual_sizes == expected_sizes, f"Unexpected split sizes: {actual_sizes}"

split_ids = {
    "train": set(train["source_row_id"]),
    "validation": set(validation["source_row_id"]),
    "test": set(test["source_row_id"]),
}
assert split_ids["train"].isdisjoint(split_ids["validation"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["validation"].isdisjoint(split_ids["test"])
assert set.union(*split_ids.values()) == set(structured["source_row_id"])

for split_name, frame in [("train", train), ("validation", validation), ("test", test)]:
    print(
        f"{split_name:10s} rows={len(frame):5d} "
        f"positive proportion={frame['subscribed'].mean():.5f}"
    )

## 5. Perform EDA on training evidence

EDA should reveal patterns, anomalies, and preparation needs. Keep it purposeful: class balance, missing-information markers, the `pdays` sentinel, and a few numerical ranges. Do not inspect validation or test feature values.

**STOP:** predict whether a classifier that always says `no` would appear accurate. Calculate the approximate majority baseline from the training target proportion shown above.

> **Your prediction and calculation:**  
> 


In [ ]:
# Training target only. No model metric is calculated here.
class_counts = train["subscribed"].value_counts().sort_index()
majority_proportion = class_counts.max() / class_counts.sum()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["no (0)", "yes (1)"], class_counts.values, color=["#4776a8", "#e28f41"])
ax.set_title("Training target distribution")
ax.set_ylabel("Examples")
for position, count in enumerate(class_counts.values):
    ax.text(position, count, f" {count:,}", ha="center", va="bottom")
plt.show()
print(f"Majority-class proportion: {majority_proportion:.3f}")

In [ ]:
training_unknown_counts = (train == "unknown").sum()
training_unknown_counts = training_unknown_counts[training_unknown_counts > 0].sort_values(ascending=False)
never_contacted_count = int((train["pdays"] == 999).sum())
previously_contacted_pdays = train.loc[train["pdays"] != 999, "pdays"]

print("Training literal 'unknown' counts:")
print(training_unknown_counts.to_string())
print(f"\npdays=999: {never_contacted_count:,} training rows")
print("pdays range when previously contacted:",
      (int(previously_contacted_pdays.min()), int(previously_contacted_pdays.max())))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(train["age"], bins=25, color="#4776a8", edgecolor="white")
axes[0].set(title="Training age distribution", xlabel="Age", ylabel="Examples")
axes[1].hist(train["campaign"], bins=30, color="#e28f41", edgecolor="white")
axes[1].set(title="Training current-campaign contacts", xlabel="Contact count")
plt.tight_layout()
plt.show()

### Turn observations into decisions

**STOP — answer from the displayed evidence and source documentation:**

1. Why will overall accuracy alone be risky in a later evaluation?
2. What information would be lost if every literal `unknown` row were deleted?
3. Why is treating 999 as an elapsed-day measurement misleading?
4. Does the long right tail of `campaign` prove that its largest observations are defects? What would you need to know?
5. Which observed preparation need belongs to preprocessing, and which proposed new field belongs to feature engineering?

> **Your answer:**  
> 
> 


## 6. Encode meaning before encoding categories

For `pdays`, one field cannot faithfully represent both elapsed days and “never contacted.” Preserve the state separately, then let the later training-fitted imputer fill the numerical absence.

For literal `unknown`, the source says deletion, imputation, or treating it as a category are possible. This lab retains it as an explicit category: deletion would discard many examples, while an imputed occupation or financial status would assert facts not present in the source.

**STOP:** explain why replacing `pdays=999` with zero **without** a separate indicator would create an ambiguity. Then complete the two policy tokens.

<details><summary>Hint</summary>The source sentinel is <code>999</code>; the selected unknown policy token is <code>explicit_category</code>.</details>

> **Your explanation:**  
> 


In [ ]:
PDAYS_SENTINEL = None   # TODO
UNKNOWN_POLICY = None   # TODO

assert PDAYS_SENTINEL == 999, "Use the value documented by UCI."
assert UNKNOWN_POLICY == "explicit_category", "Record the policy chosen above."

def prepare_semantic_frame(frame):
    prepared = frame.copy()
    prepared["previously_contacted"] = (prepared["pdays"] != PDAYS_SENTINEL).astype("int8")
    prepared["pdays"] = prepared["pdays"].replace(PDAYS_SENTINEL, np.nan)
    return prepared

prepared_train = prepare_semantic_frame(train)
prepared_validation = prepare_semantic_frame(validation)
prepared_test = prepare_semantic_frame(test)

for frame in (prepared_train, prepared_validation, prepared_test):
    assert not (frame["pdays"] == PDAYS_SENTINEL).any()
    assert set(frame["previously_contacted"].unique()) == {0, 1}
    assert frame.loc[frame["previously_contacted"] == 0, "pdays"].isna().all()

print("Semantic transformation applied consistently to all three splits.")

## 7. Define model inputs and learned preprocessing

The prepared semantic table remains understandable to humans. The model pipeline will learn median values, scales, and one-hot columns from training data only, then repeat those transformations for later inputs.

**STOP:** list the categorical inputs. Do not include the target or metadata. Then predict why logistic regression benefits from scaling numerical features even though a decision tree generally does not.

<details><summary>API hint</summary>Compare your list with <code>prepared_train.select_dtypes(exclude="number")</code>. There should be ten categorical inputs.</details>

> **Your categorical list and scaling explanation:**  
> 


In [ ]:
CATEGORICAL_FEATURES = None  # TODO: list the ten categorical model inputs

METADATA_COLUMNS = ["source_row_id"]
TARGET_COLUMN = "subscribed"
FULL_FEATURES = [
    column for column in prepared_train.columns
    if column not in METADATA_COLUMNS + [TARGET_COLUMN]
]
expected_categorical = prepared_train[FULL_FEATURES].select_dtypes(exclude="number").columns.tolist()
assert CATEGORICAL_FEATURES == expected_categorical, (
    "Recheck the displayed dtypes and preserve source-column order."
)
NUMERIC_FEATURES = [column for column in FULL_FEATURES if column not in CATEGORICAL_FEATURES]

assert len(FULL_FEATURES) == 20
assert len(CATEGORICAL_FEATURES) == 10
assert len(NUMERIC_FEATURES) == 10
assert set(FULL_FEATURES) == set(CATEGORICAL_FEATURES) | set(NUMERIC_FEATURES)
assert not set(CATEGORICAL_FEATURES) & set(NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
print("Numerical features:", NUMERIC_FEATURES)

In [ ]:
# Build and inspect the reusable preprocessor. No classifier is evaluated here.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

X_train = prepared_train[FULL_FEATURES]
X_validation = prepared_validation[FULL_FEATURES]
y_train = prepared_train[TARGET_COLUMN]

preprocessing_check = clone(preprocessor).fit(X_train)
transformed_train = preprocessing_check.transform(X_train)
transformed_validation = preprocessing_check.transform(X_validation)

assert transformed_train.shape[0] == len(prepared_train)
assert transformed_validation.shape[0] == len(prepared_validation)
assert transformed_train.shape[1] == transformed_validation.shape[1]
assert not np.isnan(transformed_train.data if hasattr(transformed_train, "data") else transformed_train).any()

print("Semantic input columns:", len(FULL_FEATURES))
print("Transformed model columns learned from training:", transformed_train.shape[1])

### Explain the pipeline boundary

**STOP:** answer each with the component name and its role.

1. Which component learns a numerical replacement value?
2. Which component makes numerical magnitudes comparable for logistic regression?
3. Which component converts categories into model inputs?
4. What happens if an operational row contains a category absent from training?
5. Why should the whole fitted pipeline be saved instead of only the classifier?

> **Your answer:**  
> 
> 


## 8. Complete the applicable-activity ledger

Before creating artifacts, record whether each section 3.2.1 activity was **performed**, **deferred**, or **not applicable**, with concrete evidence. Include at least: acquisition, cleaning, anonymization, format transformation, imputation, scaling, augmentation, sampling, feature selection/extraction, and EDA.

Be careful with wording: median imputation and scaling are configured and will be learned from training by the pipeline; augmentation and sampling were not justified; no direct identifier was supplied to anonymize, while the exercise-created row ID is metadata only.

> **Your activity ledger:**  
> 
> | Activity | Performed, deferred, or not applicable | Evidence and reason |
> | --- | --- | --- |
> | Data acquisition | | |
> | Cleaning | | |
> | Anonymization/removal | | |
> | Format transformation | | |
> | Imputation | | |
> | Scaling/normalization | | |
> | Augmentation | | |
> | Sampling | | |
> | Feature selection/extraction | | |
> | EDA | | |

## 9. Create the downstream handoff

The code below is supplied scaffolding. It trains one predetermined logistic-regression pipeline on training rows and writes the prepared semantic data, frozen split membership, model, and manifest. It deliberately makes no predictions and calculates no model metrics.

**STOP:** confirm that your ledger is complete. Predict which four files will be created and explain why the manifest is needed in addition to the CSV and model. Then change the confirmation to `True`.

> **Your prediction and manifest explanation:**  
> 


In [ ]:
I_COMPLETED_THE_ACTIVITY_LEDGER = False  # TODO: change only after writing above
assert I_COMPLETED_THE_ACTIVITY_LEDGER, "Complete the preparation ledger before creating the handoff."

baseline_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        C=1.0,
        max_iter=2000,
        class_weight=None,
    )),
])
baseline_model.fit(X_train, y_train)

prepared_data = pd.concat(
    [prepared_train, prepared_validation, prepared_test],
    ignore_index=True,
).sort_values("source_row_id")
prepared_data = prepared_data[["source_row_id", *FULL_FEATURES, TARGET_COLUMN]]

split_assignments = pd.concat([
    train[["source_row_id"]].assign(split="train"),
    validation[["source_row_id"]].assign(split="validation"),
    test[["source_row_id"]].assign(split="test"),
], ignore_index=True).sort_values("source_row_id")

assert len(prepared_data) == 41_188
assert prepared_data["source_row_id"].is_unique
assert split_assignments["source_row_id"].is_unique
assert set(prepared_data["source_row_id"]) == set(split_assignments["source_row_id"])
assert "duration" not in prepared_data.columns

ARTIFACTS_DIR = LAB_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
prepared_path = ARTIFACTS_DIR / "prepared_data.csv"
assignments_path = ARTIFACTS_DIR / "split_assignments.csv"
model_path = ARTIFACTS_DIR / "baseline_model.joblib"
manifest_path = ARTIFACTS_DIR / "manifest.json"

prepared_data.to_csv(prepared_path, index=False, na_rep="")
split_assignments.to_csv(assignments_path, index=False)
joblib.dump(baseline_model, model_path)

manifest = {
    "schema_version": "1.0",
    "source": {
        "dataset": "UCI Bank Marketing",
        "variant": "bank-additional-full.csv",
        "sha256": SOURCE_SHA256,
        "rows": 41_188,
    },
    "prediction_time": PREDICTION_TIME,
    "target": {
        "column": TARGET_COLUMN,
        "raw_mapping": TARGET_MAPPING,
        "positive_class": 1,
    },
    "columns": {
        "metadata": METADATA_COLUMNS,
        "full_features": FULL_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "numeric_features": NUMERIC_FEATURES,
    },
    "preparation": {
        "excluded_features": {"duration": "unavailable before the scheduled call"},
        "unknown_policy": UNKNOWN_POLICY,
        "pdays_sentinel": PDAYS_SENTINEL,
        "derived_features": ["previously_contacted"],
        "exact_matching_rows_beyond_first": exact_matches_beyond_first,
        "exact_matches_retained": KEEP_EXACT_MATCHES,
    },
    "split": {
        "method": "two stratified train_test_split calls",
        "seed": SPLIT_SEED,
        "proportions": {"train": 0.60, "validation": 0.20, "test": 0.20},
        "sizes": expected_sizes,
        "roles": {
            "train": "fit preprocessing and model parameters",
            "validation": "evaluate baseline and compare development candidates",
            "test": "one final check after HO-3.3.3 selection",
        },
    },
    "baseline_model": {
        "artifact": model_path.name,
        "pipeline_steps": ["preprocessor", "classifier"],
        "classifier": "LogisticRegression",
        "parameters": {
            "solver": "lbfgs", "C": 1.0, "max_iter": 2000, "class_weight": None
        },
        "trained_on": "train",
        "performance_metrics_included": False,
    },
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

trusted_reload = joblib.load(model_path)
assert list(trusted_reload.named_steps) == ["preprocessor", "classifier"]
for artifact_path in [prepared_path, assignments_path, model_path, manifest_path]:
    assert artifact_path.exists() and artifact_path.stat().st_size > 0
    print(f"Created {artifact_path.relative_to(LAB_ROOT)}")
print("Baseline trained and saved. No validation or test predictions were made.")

## Completion checkpoint

You are finished with the notebook when you can do all of these without reading code line by line:

- [ ] Point to evidence for every preparation decision.
- [ ] Distinguish literal missing-information categories, numerical missing values, and sentinels.
- [ ] Explain why `duration` is invalid despite being predictive.
- [ ] Explain why learned preprocessing was fitted on training rows only.
- [ ] Map the work to acquisition, preprocessing, feature engineering, and EDA.
- [ ] Confirm that all four handoff artifacts exist.
- [ ] State that model quality has not yet been evaluated.

Now close or minimize the notebook and complete `learning_log.md` from memory. Return here only afterward to correct specific gaps. The next exercise may use validation data to calculate performance metrics; the test set remains sealed until the final model/dataset comparison exercise.